(net-selectivity-notebook)=
# Correct proportions for net selectivity


In [ ]:
%run ./number_proportions.ipynb

## Import necessary modules

With the number proportions calculated, we can optionally correct them for length-based net selectivity. This is done via the `selectivity` module of the `survey` sub-package.

In [4]:
from echopop.survey import selectivity

## Correcting number proportions

The observed proportions are corrected using a logistic selectivity model. The `correct_number_proportions` function evaluates selectivity at each length bin $\ell$, applies inverse probability weighting, and renormalizes the number proportion within each stratum. It has three arguments:

- `number_data`: An `xarray.Dataset` (or dictionary of `xarray.Dataset` objects) representing the overall number proportions for fish. 
- `selectivity_params`: A dictionary of selectivity parameters that comprises:
    - **Exactly one** parameter pair:
        - Intercept-slope
            - `intercept`: Logistic regression intercept coefficient ($\beta_0$).
            - `slope`: Logistic regression slope coefficient ($\beta_1$).
        - Biological
            - `l50`: The length-at-50%-retention parameter, which is the length at which selectivity is 0.5.
            - `sr`: The selection range, which is the length interval between 25% and 75% retention.
    - `minimum_selectivity`: A lower bound applied to selectivity values to avoid division-by-zero errors. 
- `stratum_dim`: Dimensions that define the normalization groups. When provided, both observed and corrected length proportions are normalized to sum to 1 within each stratum.

This is ran via:

In [5]:
# Net selectivity parameterization
NET_SELECTIVITY = {
    "l50": 10.9,
    "sr": 14.0,
    "minimum_selectivity": 1e-12
}

# Apply net selectivity corrections
ds_number_proportions_corrected = selectivity.correct_number_proportions(
    number_data=dict_ds_number_proportion,
    selectivity_params=NET_SELECTIVITY,
    stratum_dim=["stratum_ks"]
)


This results in a single `xarray.Dataset`:

In [6]:
ds_number_proportions_corrected

<xarray.Dataset> Size: 778kB
Dimensions:                           (stratum_ks: 9, length_bin: 40, sex: 3,
                                       age_bin: 22)
Coordinates:
  * stratum_ks                        (stratum_ks) int64 72B 0 1 2 3 4 5 6 7 8
  * length_bin                        (length_bin) object 320B (1.0, 3.0] ......
  * sex                               (sex) object 24B 'female' 'male' 'unsexed'
  * age_bin                           (age_bin) object 176B (0.5, 1.5] ... (2...
Data variables:
    length_proportions_observed       (stratum_ks, length_bin, sex) float64 9kB ...
    length_proportions_corrected      (stratum_ks, length_bin, sex) float64 9kB ...
    length_age_proportions_observed   (stratum_ks, length_bin, age_bin, sex) float64 190kB ...
    length_age_proportions_corrected  (stratum_ks, length_bin, age_bin, sex) float64 190kB ...
    proportion_overall                (stratum_ks, length_bin, age_bin, sex) float64 190kB ...
    proportion                        (stratum_ks, length_bin, age_bin, sex) float64 190kB ...

Since there were aged *and* unaged data in the dictionary, the coordinates include `age_bin`. If this correction was run for only the unaged data, then the `age_bin` coordinate would not be included. The variables in this output are:

- `length_proportions_observed`: The original, uncorrected overall number proportions distributed across just length.
- `length_proportions_corrected`: The corrected overall number proportions distributed across length and age.
- `length_age_proportions_observed`: The original, uncorrected overall number proportions distributed across length and age.
- `length_age_proportions_observed`: The corrected overall number proportions distributed across length and age.
- `proportion` and `proportion_overall`: These correspond to the same outputs contained in the `xarray.Dataset` produced by the `number_proportions` function.

## Converting to corrected weight proportions

The corrected number proportions are converted to weight proportions by multiplying the corrected number proportions by the mean length-binned weights. The `correct_weight_proportions` function has three arguments:

- `corrected_number_proportions`: The above corrected number proportions `xarray.Dataset`.
- `mean_length_binned_weights`: The mean length-binned weights aligned on `length_bin`.
- `stratum_dim`: Dimensions that define the normalization groups. When provided, both observed and corrected length proportions are normalized to sum to 1 within each stratum.

This is ran via:

In [7]:
# Derive weight proportioms from selectivity-corrected number proportions using mean length-binned weights
ds_weight_proportions_corrected = selectivity.correct_weight_proportions(
    corrected_number_proportions=ds_number_proportions_corrected,
    mean_length_binned_weights=da_binned_weight_table,
    stratum_dim=["stratum_ks"]
)

Similar to `correct_number_proportions`, the `correct_weight_proportions` yields an `xarray.Dataset` with identical variables:

In [8]:
ds_weight_proportions_corrected

<xarray.Dataset> Size: 778kB
Dimensions:                           (stratum_ks: 9, length_bin: 40, sex: 3,
                                       age_bin: 22)
Coordinates:
  * stratum_ks                        (stratum_ks) int64 72B 0 1 2 3 4 5 6 7 8
  * length_bin                        (length_bin) object 320B (1.0, 3.0] ......
  * sex                               (sex) object 24B 'female' 'male' 'unsexed'
  * age_bin                           (age_bin) object 176B (0.5, 1.5] ... (2...
Data variables:
    length_proportions_observed       (stratum_ks, length_bin, sex) float64 9kB ...
    length_proportions_corrected      (stratum_ks, length_bin, sex) float64 9kB ...
    length_age_proportions_observed   (stratum_ks, length_bin, age_bin, sex) float64 190kB ...
    length_age_proportions_corrected  (stratum_ks, length_bin, age_bin, sex) float64 190kB ...
    proportion                        (stratum_ks, length_bin, age_bin, sex) float64 190kB ...
    proportion_overall                (stratum_ks, length_bin, age_bin, sex) float64 190kB ...

## Integrating into general workflow

Once calculated, both the corrected number and weight proportions can be used for all downstream functions.